# STEP 9: CNN (TensorFlow / Keras).
A LeNet-style 1D convolutional network over the 25-frame sequences:
Conv -> Pool -> Conv -> Pool -> Flatten -> Dense -> Dense -> output
Tuned by grid search on a 60/20/20 split; only the winner is scored on TEST.
Input:  data/features/noxi_sequences.npy  (windows, 25, 802)
data/features/noxi_engagement.parquet
Output: reports/cnn_tuning_results.csv + reports/noxi_cnn_tuned.keras

*Notebook version of `src/09_cnn.py`: same code, split into cells. Running it does exactly what the script does.*

### Dataset
**NoXi** (NOvice eXpert Interaction) - multimodal recordings of expert-novice conversations with continuous engagement labels.
- Dataset page (MultiMediate): https://multimediate.perceptualui.org/datasets/Dataset_NoXi/ (if it is offline, use the links below)
- Paper: A. Cafaro et al., "The NoXi Database: Multimodal Recordings of Mediated Novice-Expert Interactions", ACM ICMI 2017: https://eprints.nottingham.ac.uk/45827 (DOI https://doi.org/10.1145/3136755.3136780)
- Extension NoXi+J (2024): https://arxiv.org/abs/2409.13726
- Data access: University of Augsburg, Human-Centered AI lab - noxi@hcai.eu
- Used here: 9 conversations (English 027, 028, 030; French 001, 013, 021; German 070, 072, 077) from the NoXi train set:
  eGeMAPS voice features, OpenFace 2.0 face features, engagement labels (25 per second) and the pretrained-Transformer embeddings shipped with the corpus.
- Licence: NoXi is available for research under an End-User Licence Agreement (signed by the supervisor). The data is **not included** here and must not be shared;
  request access on the dataset page and place the downloaded zips as described in `01_extract_data`.

Go to the project folder.

In [ ]:
import os, sys
if os.getcwd().endswith("notebooks"):
    os.chdir("..")              # work from the project folder, like the scripts
sys.path.insert(0, "src")       # so helper files in src/ (e.g. splits.py) can be imported

silence TensorFlow (must be set before it is imported)

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)

import itertools
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

keras.utils.set_random_seed(42)
FEAT = os.path.join("data", "features", "noxi_engagement.parquet")
SEQ = os.path.join("data", "features", "noxi_sequences.npy")
os.makedirs("reports", exist_ok=True)
EPOCHS = 15          # per config during the search

metric: 1.0 = perfect

In [ ]:
def ccc(yt, yp):
    yt, yp = np.asarray(yt), np.asarray(yp)
    cov = ((yt - yt.mean()) * (yp - yp.mean())).mean()
    return (2 * cov) / (yt.var() + yp.var() + (yt.mean() - yp.mean()) ** 2 + 1e-9)

load the 25-frame sequences and the labels, split 60/20/20 (same seeds as step 7)

In [ ]:
y = pd.read_parquet(FEAT, columns=["engagement"])["engagement"].to_numpy("float32")
X = np.load(SEQ)
assert len(X) == len(y), "sequence file and table are out of sync - rerun 02_build_features.py"
n_steps, n_feat = X.shape[1], X.shape[2]
i_tmp, i_te = train_test_split(np.arange(len(y)), test_size=0.2, random_state=42)
i_tr, i_va = train_test_split(i_tmp, test_size=0.25, random_state=42)

scale every frame with a scaler fitted on TRAINING frames only

In [ ]:
sc = StandardScaler().fit(X[i_tr].reshape(-1, n_feat))

Function `prep`.

In [ ]:
def prep(idx):
    return sc.transform(X[idx].reshape(-1, n_feat)).reshape(len(idx), n_steps, n_feat).astype("float32")

Show the results.

In [ ]:
X_tr, X_va, X_te = prep(i_tr), prep(i_va), prep(i_te)
y_tr, y_va, y_te = y[i_tr], y[i_va], y[i_te]
del X
print(f"train {len(X_tr)} | val {len(X_va)} | test {len(X_te)} | input {n_steps}x{n_feat}\n", flush=True)

LeNet-style 1D-CNN: two conv+pool blocks, flatten, two dense layers, output

In [ ]:
def build(filters, kernel, drop):
    return keras.Sequential([
        layers.Input(shape=(n_steps, n_feat)),
        layers.Conv1D(filters, kernel, activation="relu"), layers.MaxPooling1D(2),
        layers.Conv1D(filters * 2, kernel, activation="relu"), layers.MaxPooling1D(2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"), layers.Dropout(drop),
        layers.Dense(64, activation="relu"), layers.Dropout(drop),
        layers.Dense(1),
    ])

build, train and score ONE configuration on validation (seed reset so all start equal)

In [ ]:
def run(filters, kernel, drop, epochs=EPOCHS):
    keras.utils.set_random_seed(42)
    m = build(filters, kernel, drop)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    m.fit(X_tr, y_tr, epochs=epochs, batch_size=128, verbose=0)
    return ccc(y_va, m.predict(X_va, verbose=0).flatten()), m

grid search: 2 filter sizes x 2 kernel sizes x 2 dropouts = 8 configs

In [ ]:
FILTERS, KERNELS, DROPS = [32, 64], [3, 5], [0.1, 0.3]
rows = []
print(f"{'filters':>8} {'kernel':>7} {'dropout':>8} {'val CCC':>8}", flush=True)

Main loop.

In [ ]:
for f, k, d in itertools.product(FILTERS, KERNELS, DROPS):
    val, _ = run(f, k, d)
    rows.append(dict(filters=f, kernel=k, dropout=d, val_ccc=val))
    print(f"{f:8d} {k:7d} {d:8.1f} {val:8.3f}", flush=True)

rank and save

In [ ]:
res = pd.DataFrame(rows).sort_values("val_ccc", ascending=False).reset_index(drop=True)
res.to_csv(os.path.join("reports", "cnn_tuning_results.csv"), index=False)
print("\n=========== CNN TUNING RESULTS (ranked by validation CCC) ===========")
print(res.to_string(index=False))

retrain the winner for longer, then score it on the untouched test set

In [ ]:
w = res.iloc[0]
val, model = run(int(w["filters"]), int(w["kernel"]), float(w["dropout"]), epochs=30)
pred = model.predict(X_te, verbose=0).flatten()

print("\n=========== BEST CNN ===========")
print(f"  filters   : {int(w['filters'])} -> {int(w['filters']) * 2}")
print(f"  kernel    : {int(w['kernel'])}")
print(f"  dropout   : {w['dropout']}")
print(f"  validation CCC: {val:.3f}")
print(f"  TEST CCC      : {ccc(y_te, pred):.3f}")
print(f"  TEST Pearson  : {np.corrcoef(y_te, pred)[0, 1]:.3f}")
print(f"  TEST MAE      : {mean_absolute_error(y_te, pred):.3f}")

print("\n============ ARCHITECTURE ============")
model.summary()

model.save(os.path.join("reports", "noxi_cnn_tuned.keras"))
print("\nSaved -> reports/cnn_tuning_results.csv, reports/noxi_cnn_tuned.keras")